# RVEWeb on Google Colab (NVIDIA T4)

This notebook deploys RVEWeb natively (no Docker) on a Colab GPU runtime and exposes the console through a public URL.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

What it does:
- installs FFmpeg, the NVIDIA Vulkan driver, and Python dependencies;
- downloads the pinned RIFE/Real-ESRGAN models;
- starts the FastAPI app on port 8000;
- uses the T4 through Vulkan for Real-ESRGAN / RIFE 4.6–4.26 and through CUDA ONNX for RIFE 4.27;
- runs a small GPU smoke test and prints both a Colab iframe URL and a public Cloudflare URL.

**Security:** the app has no authentication. Anyone with the public URL can submit jobs while the runtime is alive. Stop the runtime when finished.

**Persistence:** `/content` is ephemeral. The library and queue disappear when the Colab runtime is recycled. Download anything you want to keep.

## 1. Confirm the T4 GPU

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    raise SystemExit('No CUDA GPU. Select Runtime → Change runtime type → T4 GPU, then rerun.')
name = torch.cuda.get_device_name(0)
print('GPU:', name)
if 'T4' not in name:
    print('Warning: this notebook was tuned for a T4; continuing with', name)

## 2. Install FFmpeg, Vulkan, and the matching NVIDIA ICD

Colab already has the CUDA driver. Vulkan needs the userspace GL/Vulkan driver package matching that driver. The next cell installs it and verifies that `vulkaninfo` sees the T4.

In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq ffmpeg unzip curl libvulkan1 vulkan-tools libgl1 libglib2.0-0 ca-certificates
DRIVER_MAJOR=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -1 | cut -d. -f1)
echo "NVIDIA driver major: $DRIVER_MAJOR"
apt-get install -y -qq "libnvidia-gl-${DRIVER_MAJOR}" || true
if ! vulkaninfo --summary 2>/dev/null | grep -qi 'Tesla T4'; then
  for v in 580 570 550 540 535 525 510; do
    if apt-get install -y -qq "libnvidia-gl-${v}" 2>/dev/null; then break; fi
  done
fi
echo '--- Vulkan summary ---'
vulkaninfo --summary 2>&1 | sed -n '1,80p'


In [ ]:
import os, re, subprocess
result = subprocess.run(['vulkaninfo', '--summary'], capture_output=True, text=True)
out = result.stdout + result.stderr
devices, current = [], None
for line in out.splitlines():
    s = line.strip()
    m = re.match(r'GPU(\d+):$', s)
    if m:
        if current: devices.append(current)
        current = {'index': int(m.group(1))}
    elif current is not None and '=' in s:
        k, v = s.split('=', 1)
        current[k.strip()] = v.strip()
if current: devices.append(current)
print(devices)
t4 = [d for d in devices if 'Tesla T4' in d.get('deviceName', '')]
GPU_DEVICE = str(t4[0]['index'] if t4 else 0)
os.environ['GPU_DEVICE'] = GPU_DEVICE
print('VULKAN_T4 =', bool(t4), '| GPU_DEVICE =', GPU_DEVICE)
if not t4:
    print('WARNING: Vulkan cannot see the T4. Real-ESRGAN will fall back to Lanczos; RIFE 4.27 can still use CUDA.')

## 3. Clone the repository

If you publish the repo under a different owner/name, edit `REPO_URL`. For a private repo, use a GitHub token in the URL.

In [ ]:
%%bash
set -e
REPO_URL='https://github.com/Epickitrolaz/rveweb.git'
rm -rf /content/rveweb
git clone --depth 1 "$REPO_URL" /content/rveweb
cd /content/rveweb && git rev-parse --short HEAD && ls


## 4. Install Python dependencies

We replace the CPU `onnxruntime` wheel from `requirements.txt` with `onnxruntime-gpu` so RIFE 4.27 can run on the T4 through CUDA.

In [ ]:
%%bash
set -e
cd /content/rveweb
python -m pip uninstall -y -q onnxruntime onnxruntime-gpu || true
grep -v '^onnxruntime==' backend/requirements.txt > /tmp/rveweb-requirements-colab.txt
python -m pip install -q -r /tmp/rveweb-requirements-colab.txt
python -m pip install -q 'onnxruntime-gpu==1.32.0'
python - <<'PY'
import onnxruntime as ort
print('onnxruntime', ort.__version__)
print('providers', ort.get_available_providers())
PY

## 5. Download pinned models

This downloads RIFE 4.27 ONNX, the RIFE 4.6–4.26 Vulkan models, and the Real-ESRGAN ncnn models (~180 MB).

In [ ]:
%%bash
set -e
cd /content/rveweb
bash backend/scripts/fetch_models.sh /content/rveweb/backend/models
du -sh /content/rveweb/backend/models/*


## 6. Start the app

In [ ]:
import json, os, subprocess, sys, time, urllib.request
os.makedirs('/content/rveweb-data', exist_ok=True)
env = os.environ.copy()
env.update({
    'DATA_DIR': '/content/rveweb-data',
    'MODEL_DIR': '/content/rveweb/backend/models',
    'FRONTEND_DIR': '/content/rveweb/frontend',
    'GPU_DEVICE': os.environ.get('GPU_DEVICE', '0'),
    'RVE_ACCELERATOR': 'cuda',
    'RIFE_THREADS': '2',
    'FFMPEG_THREADS': '2',
    'MAX_ACTIVE_JOBS': '1',
    'LOG_LEVEL': 'INFO',
})
log = open('/content/rveweb.log', 'w')
proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000',
     '--workers', '1', '--limit-concurrency', '128', '--timeout-keep-alive', '5'],
    cwd='/content/rveweb/backend', env=env, stdout=log, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print('server pid', proc.pid)
ready = False
for _ in range(90):
    if proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as r:
            print('health', r.read().decode())
            ready = True
            break
    except Exception:
        time.sleep(2)
if not ready:
    log.flush()
    print(open('/content/rveweb.log').read()[-6000:])
    raise RuntimeError('RVEWeb did not start')

## 7. Verify T4 acceleration and run a smoke test

The output engine should report `upscaler: vulkan` and `interpolation: cuda-onnx` on a healthy T4 runtime. The test clip is 1 s of 320×180 video.

In [ ]:
import json, subprocess, time, urllib.request
import requests
BASE = 'http://127.0.0.1:8000'

def get(path):
    with urllib.request.urlopen(BASE + path, timeout=20) as r:
        return json.loads(r.read())

def put(path, payload):
    req = urllib.request.Request(BASE + path, data=json.dumps(payload).encode(),
                                 headers={'Content-Type': 'application/json'}, method='PUT')
    with urllib.request.urlopen(req, timeout=20) as r:
        return json.loads(r.read())

info = get('/api/system')
print(json.dumps(info['gpu'], indent=2))
print('ORT providers:', __import__('onnxruntime').get_available_providers())
put('/api/settings', {
    'processingMode': 'auto',
    'defaults': {
        'encode': {'codec': 'h264', 'crf': 20, 'preset': 'veryfast', 'container': 'mp4'},
        'audio': {'mode': 'copy'},
    },
})

subprocess.run([
    'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
    '-f', 'lavfi', '-i', 'testsrc2=size=320x180:rate=24:duration=1',
    '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '26', '-pix_fmt', 'yuv420p',
    '-an', '/tmp/colab-smoke.mp4'
], check=True)
settings = {
    'upscale': {'enabled': True, 'model': 'realesrgan-video', 'scale': 2},
    'interpolate': {'enabled': True, 'model': 'rife-4.27', 'factor': 2, 'maxHeight': 1080},
    'denoise': {'enabled': False, 'strength': 'medium'},
    'decompress': {'enabled': False, 'strength': 'light'},
    'encode': {'codec': 'h264', 'crf': 20, 'preset': 'veryfast', 'container': 'mp4'},
    'audio': {'mode': 'none'},
}
with open('/tmp/colab-smoke.mp4', 'rb') as f:
    response = requests.post(BASE + '/api/jobs', files={'file': f}, data={'settings': json.dumps(settings)}, timeout=60)
response.raise_for_status()
job = response.json()
print('job', job['id'])
for _ in range(90):
    job = get('/api/jobs/' + job['id'])
    print(job['status'], job['stage'], job['progress'], job.get('etaSeconds'))
    if job['status'] in ('completed', 'failed', 'cancelled'):
        break
    time.sleep(3)
if job['status'] != 'completed':
    raise RuntimeError('Smoke job failed: ' + json.dumps(job, indent=2))
engine = job['output']['engine']
print('engine:', engine)
if engine.get('interpolation') == 'cuda-onnx' and engine.get('upscaler') == 'vulkan':
    print('T4 acceleration verified: CUDA RIFE + Vulkan Real-ESRGAN')
else:
    print('WARNING: one or more accelerators fell back to CPU:', engine)

## 8. Open the console

The Colab proxy URL works while the notebook is connected and requires your Google session. If Cloudflare is reachable, a public URL is also printed.

In [ ]:
from google.colab import output
from IPython.display import display, HTML
proxy_url = output.eval_js('google.colab.kernel.proxyPort(8000)')
print('Colab proxy URL:', proxy_url)
display(HTML(f'<iframe src="{proxy_url}" width="100%" height="980" style="border:1px solid #333"></iframe>'))

In [ ]:
import os, re, subprocess, time
CF = '/content/cloudflared'
if not os.path.exists(CF):
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', CF], check=True)
    os.chmod(CF, 0o755)
cf_log = open('/content/cloudflared.log', 'w')
cf = subprocess.Popen([CF, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate', '--protocol', 'http2'], stdout=cf_log, stderr=subprocess.STDOUT, start_new_session=True)
public_url = None
for _ in range(30):
    time.sleep(1)
    text = open('/content/cloudflared.log').read()
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', text)
    if match:
        public_url = match.group(0)
        break
print('Public URL:', public_url)
if public_url:
    print('Open it from any browser while this runtime is alive. This app has no authentication.')
else:
    print('Cloudflare tunnel did not produce a URL. Use the Colab proxy iframe above. Log tail:')
    print(open('/content/cloudflared.log').read()[-2000:])

## 9. Stop the station

Run this cell when finished, or stop the Colab runtime.

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'uvicorn app.main:app'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
print('stopped')